# Session 2: LangChain Development & Tool Integration (Student Notebook)

Welcome to Session 2! In this notebook, you will learn LangChain -- the most popular framework for building LLM applications. You will master its core abstractions, compose chains using LCEL, create custom tools, and build a simple RAG pipeline.

**Consulting Context:** All examples in this session use McKinsey-style consulting scenarios -- market sizing, competitor analysis, client engagement workflows, and strategic frameworks like MECE, value chain analysis, and the growth-share matrix.

## Learning Objectives

By the end of this session, you will be able to:

1. **Use ChatModels and PromptTemplates** as LangChain's core building blocks
2. **Compose chains with LCEL** using the pipe operator (`|`)
3. **Create custom tools** using the `@tool` decorator
4. **Load and split documents** for retrieval workflows
5. **Build a RAG pipeline** that grounds LLM answers in external knowledge
6. **Manage conversation memory** across multi-turn interactions

In [ ]:
from dotenv import load_dotenv
load_dotenv()  # Load environment variables from .env

# LangSmith tracing configuration
os.environ["LANGCHAIN_TRACING_V2"] = os.getenv("LANGCHAIN_TRACING_V2", "false")
os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGCHAIN_PROJECT", "mckinsey-genai-3day")

# ============================================================
# Imports and Setup
# ============================================================

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_core.tools import tool
from langchain_core.documents import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
import json
import os

# Ensure your OpenAI API key is set:
#   export OPENAI_API_KEY="sk-..."

print("All imports successful!")

---
## Demos (Follow Along)

The following five demos are fully coded. Run each cell, observe the output, and make sure you understand what is happening before moving on.

### Demo 1: LangChain Basics — ChatModels and PromptTemplates

ChatModels wrap LLM APIs into a unified interface. PromptTemplates let you parameterize prompts with variables so they become reusable across different inputs.

**Consulting scenario:** A McKinsey engagement team needs to quickly generate structured analyses across different industries and strategy frameworks. PromptTemplates let us build reusable "analysis templates" that any consultant can invoke with different client parameters.

In [ ]:
# Demo 1 - LangChain Basics: ChatModels and PromptTemplates

# Step 1: Initialize a ChatModel
llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"), temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0.7")))

# Step 2: Simple invocation with a string
response = llm.invoke("What is McKinsey's MECE framework in one sentence?")
print("Simple invocation:")
print(f"  Type: {type(response)}")
print(f"  Content: {response.content}")

# Step 3: Invocation with message objects
messages = [
    SystemMessage(content="You are a McKinsey senior partner. Be concise and strategic."),
    HumanMessage(content="What is a value chain analysis?")
]
response = llm.invoke(messages)
print(f"\nWith messages:\n  {response.content}")

# Step 4: PromptTemplates for reusable consulting prompts
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a McKinsey consultant specializing in {domain}. Answer in {style} style."),
    ("human", "{question}")
])

formatted = prompt.format_messages(
    domain="growth strategy and market entry",
    style="executive briefing",
    question="What are the key drivers of growth in the EV battery market?"
)
print(f"\nFormatted prompt messages: {len(formatted)}")
for msg in formatted:
    print(f"  [{msg.type}]: {msg.content[:80]}...")

response = llm.invoke(formatted)
print(f"\nResponse: {response.content[:200]}...")

### Demo 2: Building Chains with LCEL (Pipe Operator)

LCEL (LangChain Expression Language) uses the `|` operator to compose components into chains. The pattern is: `prompt | model | parser`. Each component implements the Runnable interface, so they can be freely combined.

**Consulting scenario:** McKinsey engagement teams need multi-step analysis pipelines: research a market, analyze competitive dynamics, then produce an executive summary. LCEL lets us chain these steps cleanly -- mirroring how a real consulting workflow flows from data gathering through synthesis to recommendation.

In [ ]:
# Demo 2 - Building Chains with LCEL

llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"), temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0.5")))

# Step 1: Simple chain: prompt -> LLM -> string output
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a McKinsey strategy consultant. Provide clear, structured insights."),
    ("human", "Explain {concept} in exactly {num_sentences} sentences, using a business consulting context.")
])

chain = prompt | llm | StrOutputParser()

result = chain.invoke({"concept": "Porter's Five Forces", "num_sentences": "3"})
print("Simple chain result:")
print(result)

# Step 2: Chain with JSON output
json_prompt = ChatPromptTemplate.from_messages([
    ("system", "Output a JSON object with keys: framework_name, definition, consulting_use_case, complexity (1-10)."),
    ("human", "Describe the consulting framework: {concept}")
])

json_chain = json_prompt | llm | JsonOutputParser()

result = json_chain.invoke({"concept": "MECE principle"})
print(f"\nJSON chain result (type={type(result).__name__}):")
print(json.dumps(result, indent=2))

# Step 3: Multi-step chains (research -> executive summary -> translation for global offices)
summarize_prompt = ChatPromptTemplate.from_template(
    "Summarize this market research finding in one executive-ready sentence: {text}"
)
translate_prompt = ChatPromptTemplate.from_template(
    "Translate this executive summary to French for our Paris office: {text}"
)

summary_chain = summarize_prompt | llm | StrOutputParser()
translate_chain = translate_prompt | llm | StrOutputParser()

text = "The global electric vehicle battery market is projected to grow at a CAGR of 19.2% from 2024 to 2030. Key growth drivers include government incentives, declining lithium-ion costs, and rising consumer demand for sustainable transportation. Asia-Pacific dominates with 68% market share, led by CATL and BYD."

summary = summary_chain.invoke({"text": text})
print(f"\nExecutive Summary: {summary}")

translation = translate_chain.invoke({"text": summary})
print(f"French (for Paris office): {translation}")

### Demo 3: Creating and Using Custom Tools

Tools let LLMs interact with the real world. In LangChain, use the `@tool` decorator to turn any Python function into a tool that an LLM can discover and invoke. The docstring becomes the tool's description.

**Consulting scenario:** McKinsey consultants rely on specialized analytical tools daily -- market sizing calculators, benchmark databases, financial ratio analyzers. Here we build custom tools that mirror these consulting utilities and bind them to an LLM so it can decide which tool to use based on a client question.

In [ ]:
# Demo 3 - Creating and Using Custom Tools

# Step 1: Define custom consulting tools
@tool
def market_sizing_tool(population: float, adoption_rate: float, avg_revenue_per_user: float) -> dict:
    """Estimate total addressable market (TAM) given population, adoption rate (0-1), and average revenue per user."""
    tam = population * adoption_rate * avg_revenue_per_user
    return {
        "total_addressable_market": tam,
        "population": population,
        "adoption_rate_pct": f"{adoption_rate * 100:.1f}%",
        "arpu": avg_revenue_per_user,
        "formatted_tam": f"${tam:,.0f}"
    }

@tool
def benchmark_lookup_tool(metric: str) -> str:
    """Look up industry benchmark values for common consulting metrics."""
    benchmarks = {
        "saas_gross_margin": "70-85% gross margin is typical for SaaS companies",
        "retail_operating_margin": "3-5% operating margin is typical for retail",
        "pharma_r_and_d_spend": "15-25% of revenue is typical pharma R&D spend",
        "tech_revenue_growth": "20-40% YoY revenue growth for high-growth tech",
        "healthcare_ebitda_margin": "15-25% EBITDA margin for healthcare services",
    }
    return benchmarks.get(metric.lower(), f"No benchmark found for '{metric}'. Available: {list(benchmarks.keys())}")

@tool
def financial_ratio_tool(revenue: float, costs: float, assets: float) -> dict:
    """Calculate key financial ratios used in consulting engagements."""
    gross_margin = (revenue - costs) / revenue if revenue else 0
    asset_turnover = revenue / assets if assets else 0
    return {
        "gross_margin": f"{gross_margin:.1%}",
        "asset_turnover": f"{asset_turnover:.2f}x",
        "profit": f"${revenue - costs:,.0f}"
    }

# Step 2: Inspect tool metadata
print("Tool: market_sizing_tool")
print(f"  Name: {market_sizing_tool.name}")
print(f"  Description: {market_sizing_tool.description}")

# Step 3: Invoke tools directly
print(f"\nmarket_sizing_tool (US EV market): {market_sizing_tool.invoke({'population': 330_000_000, 'adoption_rate': 0.08, 'avg_revenue_per_user': 45000})}")
print(f"benchmark_lookup_tool: {benchmark_lookup_tool.invoke('saas_gross_margin')}")
print(f"financial_ratio_tool: {financial_ratio_tool.invoke({'revenue': 50_000_000, 'costs': 32_000_000, 'assets': 80_000_000})}")

# Step 4: Bind tools to an LLM
llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"), temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0")))
llm_with_tools = llm.bind_tools([market_sizing_tool, benchmark_lookup_tool, financial_ratio_tool])

response = llm_with_tools.invoke("What is the typical gross margin for a SaaS company?")
print(f"\nLLM response with tools:")
print(f"  Content: {response.content}")
print(f"  Tool calls: {response.tool_calls}")

### Demo 4: Document Loading and Text Splitting

RAG starts with loading documents and splitting them into manageable chunks. The `RecursiveCharacterTextSplitter` splits at natural boundaries (paragraphs, sentences) and maintains overlap between chunks so context is not lost at boundaries.

**Consulting scenario:** McKinsey maintains vast knowledge bases -- industry reports, client case studies, best practice documents. Before we can retrieve relevant insights, we need to split these documents into chunks that fit within LLM context windows while preserving analytical coherence.

In [ ]:
# Demo 4 - Document Loading and Text Splitting

# Step 1: Create sample McKinsey consulting documents
documents = [
    Document(
        page_content="""McKinsey Global Institute research shows that generative AI could add $2.6 to $4.4 trillion
annually to the global economy across 63 use cases. The banking, high-tech, and life sciences
sectors stand to benefit most, with potential productivity gains of 3-5% of total industry revenue.
Organizations that move quickly on adoption will capture disproportionate value.""",
        metadata={"source": "mgi_genai_report.pdf", "chapter": "Executive Summary"}
    ),
    Document(
        page_content="""Post-merger integration (PMI) remains the most critical phase of any M&A transaction.
McKinsey research indicates that 70% of mergers fail to achieve expected synergies, primarily due to
cultural misalignment and slow integration execution. Best-practice PMI follows a 100-day plan
covering Day 1 readiness, synergy capture, operating model design, and cultural integration.""",
        metadata={"source": "mckinsey_ma_playbook.pdf", "chapter": "Post-Merger Integration"}
    )
]

print(f"Loaded {len(documents)} consulting documents")
for doc in documents:
    print(f"  Source: {doc.metadata['source']} | Chapter: {doc.metadata['chapter']} | Length: {len(doc.page_content)} chars")

# Step 2: Split documents into chunks
splitter = RecursiveCharacterTextSplitter(
    chunk_size=int(os.getenv("CHUNK_SIZE", "200")),
    chunk_overlap=int(os.getenv("CHUNK_OVERLAP", "30")),
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = splitter.split_documents(documents)
print(f"\nSplit into {len(chunks)} chunks:")
for i, chunk in enumerate(chunks):
    print(f"\n  Chunk {i+1} ({len(chunk.page_content)} chars) [from {chunk.metadata['source']}]:")
    print(f"    {chunk.page_content[:100]}...")

# Step 3: Show how overlap works
print("\nOverlap demonstration:")
print(f"  End of chunk 1  : ...{chunks[0].page_content[-40:]}")
if len(chunks) > 1:
    print(f"  Start of chunk 2: {chunks[1].page_content[:40]}...")

### Demo 5: Building a Simple RAG Pipeline

This demo puts it all together: a knowledge base of text chunks, a simple retrieval function, and an LCEL chain that generates answers grounded in retrieved context.

**Consulting scenario:** Imagine a McKinsey knowledge management system where consultants can query the firm's repository of industry insights, frameworks, and engagement best practices. The RAG pipeline retrieves the most relevant knowledge snippets and generates a grounded, evidence-based answer -- just like a seasoned partner drawing on decades of firm knowledge.

In [ ]:
# Demo 5 - Building a Simple RAG Pipeline

llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"), temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0")))

# Step 1: McKinsey consulting knowledge base
knowledge_base = [
    "The MECE framework (Mutually Exclusive, Collectively Exhaustive) is McKinsey's foundational structuring principle. It ensures problem decomposition has no overlaps and no gaps, enabling rigorous hypothesis-driven analysis.",
    "McKinsey's Three Horizons of Growth framework helps companies allocate resources: Horizon 1 (core business), Horizon 2 (emerging opportunities), Horizon 3 (transformational initiatives).",
    "Value chain analysis, developed by Michael Porter, maps the primary and support activities that create value for customers. McKinsey consultants use it to identify cost reduction and differentiation opportunities.",
    "The McKinsey 7S Framework examines seven interdependent elements: Strategy, Structure, Systems, Shared Values, Style, Staff, and Skills. It is used for organizational effectiveness assessments.",
    "Digital transformation engagements typically follow McKinsey's 'rewire' approach: reimagine the business model, rebuild the technology foundation, and rewire the organization for speed and agility.",
    "Post-merger integration (PMI) success depends on capturing synergies within the first 100 days. McKinsey's PMI approach covers Day 1 readiness, clean rooms, synergy tracking, and cultural integration.",
]

# Step 2: Simple keyword-based retrieval
def simple_retrieve(query, k=2):
    scored = []
    query_words = set(query.lower().split())
    for chunk in knowledge_base:
        chunk_words = set(chunk.lower().split())
        overlap = len(query_words & chunk_words)
        scored.append((overlap, chunk))
    scored.sort(key=lambda x: x[0], reverse=True)
    return [chunk for _, chunk in scored[:k]]

# Step 3: RAG chain
rag_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a McKinsey knowledge management assistant. Answer based ONLY on the context from the firm's knowledge base. If not found, say 'This topic is not covered in the current knowledge base.'\n\nContext:\n{context}"),
    ("human", "{question}")
])

rag_chain = rag_prompt | llm | StrOutputParser()

# Step 4: Run consulting queries
questions = [
    "What is the MECE framework?",
    "How does McKinsey approach post-merger integration?",
    "What is the capital asset pricing model?"
]

for question in questions:
    retrieved = simple_retrieve(question)
    context = "\n".join(f"- {chunk}" for chunk in retrieved)
    answer = rag_chain.invoke({"context": context, "question": question})
    print(f"Q: {question}")
    print(f"A: {answer}\n")

---
## Tasks (Your Turn!)

Now it is your turn to practice. Each task below has a description, hints, and a code skeleton with `TODO` placeholders. Fill in the code to complete each task.

### Task 1: Build a Consulting Analysis Chain with Prompt Templates and Output Parsers

Create an LCEL chain that takes an **industry** and **analysis type** (e.g., "market entry", "competitive landscape", "cost optimization"), generates a structured consulting analysis with MECE-structured findings and a strategic recommendation, and returns the result as parsed JSON.

**Consulting scenario:** An engagement manager needs to quickly generate structured industry analyses for different sectors and strategy questions. The chain produces a standardized JSON output that can be fed into client-ready slide decks.

In [ ]:
# Task 1 - Build a Consulting Analysis Chain with Prompt Templates and Output Parsers

# TODO: Create an LCEL chain that:
# 1. Takes an industry and analysis_type as inputs
# 2. Uses a PromptTemplate with a McKinsey senior consultant system persona
# 3. Generates a structured analysis with JSON fields:
#      - industry (string)
#      - analysis_type (string)
#      - key_findings (list of 3 strings, each a MECE-structured insight)
#      - strategic_recommendation (string with a clear action item)
#      - risk_factors (list of 2 strings)
#
# Hint: Use ChatPromptTemplate.from_messages() with system and human messages
# Hint: Tell the system to output JSON and use MECE principles in the system message
# Hint: Use JsonOutputParser() as the final step in the chain

def create_consulting_analysis_chain():
    """Create a chain that generates structured consulting analyses."""
    # YOUR CODE HERE
    pass


# Test your chain
# chain = create_consulting_analysis_chain()
# result = chain.invoke({"industry": "electric vehicle batteries", "analysis_type": "market entry"})
# print(json.dumps(result, indent=2))

### Task 2: Create Custom Consulting Tools

Create three custom tools using the `@tool` decorator that mirror McKinsey analytical utilities: a **competitor analysis tool**, a **market sizing estimator** (TAM/SAM/SOM), and an **engagement ROI calculator**. Then bind them to an LLM.

**Consulting scenario:** These tools represent the analytical utilities that a McKinsey team would use during a typical engagement -- sizing markets, analyzing competitors, and projecting engagement value for the client.

In [ ]:
# Task 2 - Create Custom Consulting Tools

# TODO: Create three custom tools using the @tool decorator:
# 1. competitor_analysis_tool(company: str, competitors: list[str]) -> dict
#      - Takes a focal company and list of competitors
#      - Returns a dict with competitor_list, dimensions analyzed, methodology, and recommendation
#
# 2. market_sizing_estimate(total_population: float, serviceable_pct: float,
#                           obtainable_pct: float, avg_deal_value: float) -> dict
#      - Calculates TAM (total_population * avg_deal_value)
#      - Calculates SAM (TAM * serviceable_pct) and SOM (SAM * obtainable_pct)
#      - Returns formatted TAM/SAM/SOM values
#
# 3. engagement_roi_tool(engagement_cost: float, projected_savings: float,
#                        implementation_months: int) -> dict
#      - Calculates ROI percentage and payback period in months
#      - Returns engagement metrics with a verdict (Strong/Moderate/Marginal ROI)
#
# Then bind all tools to an LLM and test tool invocation.
#
# Hint: Use the @tool decorator from langchain_core.tools
# Hint: Include clear docstrings -- LangChain uses them as tool descriptions for the LLM
# Hint: Use llm.bind_tools([...]) to attach tools to the model

# YOUR CODE HERE


# Test your tools directly
# print(competitor_analysis_tool.invoke({"company": "Tesla", "competitors": ["BYD", "Rivian", "Lucid"]}))
# print(market_sizing_estimate.invoke({"total_population": 5_000_000, "serviceable_pct": 0.3, "obtainable_pct": 0.1, "avg_deal_value": 50_000}))
# print(engagement_roi_tool.invoke({"engagement_cost": 2_000_000, "projected_savings": 15_000_000, "implementation_months": 6}))

### Task 3: Implement a Conversational Consulting Advisor with Memory

Build a conversational chain that acts as a **McKinsey senior partner** advising a client CEO on strategic decisions. The advisor should maintain message history across turns, remembering previous discussion points about the client's situation -- just like a real partner would across multiple steering committee meetings.

**Consulting scenario:** A client CEO is exploring strategic options (e.g., M&A, market entry, digital transformation). The advisor draws on McKinsey frameworks and remembers the full engagement context.

In [ ]:
# Task 3 - Implement a Conversational Consulting Advisor with Memory

# TODO: Build a consulting advisor chain that:
# 1. Uses a system prompt positioning the LLM as a McKinsey senior partner
#    (mention MECE, 7S, Three Horizons frameworks in the system prompt)
# 2. Maintains a list of previous messages (chat history)
# 3. Uses MessagesPlaceholder in the prompt to include history
# 4. Has a chat() method that appends user/assistant messages to history
# 5. Returns the advisor's strategic response
#
# Hint: Use ChatPromptTemplate with MessagesPlaceholder("history")
# Hint: Store history as a list of HumanMessage and AIMessage objects
# Hint: Pass history to chain.invoke({"history": self.history, "input": user_input})

class ConsultingAdvisor:
    def __init__(self):
        """Initialize the consulting advisor chain with memory."""
        # YOUR CODE HERE
        pass

    def chat(self, user_input):
        """Send a message and get a strategic response, maintaining engagement history."""
        # YOUR CODE HERE
        pass

    def get_history(self):
        """Return the engagement conversation history."""
        # YOUR CODE HERE
        pass


# Test your consulting advisor
# advisor = ConsultingAdvisor()
# print(advisor.chat("We are a mid-size healthcare company considering acquiring a smaller biotech firm to expand our oncology pipeline. What should we consider?"))
# print(advisor.chat("What are the biggest post-merger integration risks we should watch for given our situation?"))
# print(f"Engagement history: {len(advisor.get_history())} messages")

### Task 4: Build a RAG-Powered Consulting Knowledge Base

Build a complete RAG system that takes McKinsey industry reports and framework documents, splits them into chunks, retrieves relevant chunks for a consulting query, and generates grounded, partner-quality answers with source attribution.

**Consulting scenario:** Build a knowledge management system that consultants can query to find relevant insights from McKinsey's proprietary industry research on EV batteries, M&A best practices, and digital banking transformation.

In [ ]:
# Task 4 - Build a RAG-Powered Consulting Knowledge Base

# TODO: Build a complete consulting RAG system that:
# 1. Takes a list of McKinsey Document objects as input
# 2. Splits them into chunks using RecursiveCharacterTextSplitter
# 3. Implements a retrieve() method that finds relevant chunks for a consulting query
# 4. Implements an ask() method that retrieves context and generates a partner-quality answer
# 5. Uses a McKinsey knowledge management assistant persona in the system prompt
# 6. Includes the source metadata in the response and structures answers using MECE principles
#
# Hint: Split documents with chunk_size=int(os.getenv("CHUNK_SIZE", "300")), chunk_overlap=int(os.getenv("CHUNK_OVERLAP", "50"))
# Hint: Use keyword matching for retrieval (score by word overlap with query)
# Hint: Include source information in the prompt context as [Source: filename]

class ConsultingRAG:
    def __init__(self, documents):
        """Initialize with McKinsey knowledge documents, split into chunks."""
        # YOUR CODE HERE
        pass

    def retrieve(self, query, k=3):
        """Retrieve the top-k most relevant chunks for a consulting query."""
        # YOUR CODE HERE
        pass

    def ask(self, question):
        """Answer a consulting question using retrieved context from the knowledge base."""
        # YOUR CODE HERE
        pass


# Test your consulting RAG system
# docs = [
#     Document(
#         page_content=(
#             "The global EV battery market is projected to reach $360 billion by 2030, growing at a CAGR of 19.2%. "
#             "Key growth drivers include government incentives (IRA in the US, Green Deal in EU), declining lithium-ion costs, "
#             "and rising consumer demand. Asia-Pacific dominates with 68% market share, led by CATL and BYD. "
#             "Supply chain concentration in China remains the primary strategic risk for Western OEMs."
#         ),
#         metadata={"source": "mckinsey_ev_battery_outlook_2024.pdf"}
#     ),
#     Document(
#         page_content=(
#             "McKinsey's analysis of 2,500 M&A transactions reveals that successful acquirers share three traits: "
#             "they conduct rigorous due diligence on cultural compatibility, they establish a dedicated integration "
#             "management office (IMO) within 30 days, and they set aggressive but achievable 100-day synergy targets. "
#             "Healthcare and technology sectors show the highest variance in post-merger outcomes, "
#             "with cultural integration being the single largest predictor of success."
#         ),
#         metadata={"source": "mckinsey_ma_excellence_report.pdf"}
#     ),
#     Document(
#         page_content=(
#             "Digital transformation across financial services is accelerating. McKinsey estimates that banks "
#             "investing in AI-driven operations can reduce costs by 20-30% while improving customer satisfaction scores "
#             "by 15-25 points. The most successful transformations follow a 'rewire' approach: reimagine customer journeys, "
#             "rebuild core technology platforms, and reorganize around agile, cross-functional teams."
#         ),
#         metadata={"source": "mckinsey_digital_banking_2024.pdf"}
#     ),
# ]
# rag = ConsultingRAG(docs)
# print(rag.ask("What are the growth drivers in the EV battery market?"))
# print()
# print(rag.ask("What makes M&A integrations successful in healthcare?"))

---
## Summary

In this session you learned the core LangChain building blocks for LLM application development, applied throughout to McKinsey consulting scenarios:

1. **ChatModels & PromptTemplates** -- How LangChain wraps LLM APIs and provides reusable, parameterized prompt templates for consulting analyses (market research, strategy frameworks, executive briefings).
2. **LCEL Chains** -- How the pipe operator (`|`) composes prompt, model, and parser into clean analysis pipelines (research to synthesis to executive summary).
3. **Custom Tools** -- How the `@tool` decorator turns Python functions into consulting tools (market sizing, competitor analysis, financial modeling, benchmark lookup) that LLMs can discover and invoke.
4. **Document Loading & Splitting** -- How to prepare McKinsey industry reports and knowledge base documents for retrieval by splitting them into overlapping chunks.
5. **RAG Pipelines** -- How to combine retrieval with generation to ground consulting answers in firm research and industry data.

**Up Next:** In Session 3, we will move from linear chains to graph-based workflows with LangGraph -- enabling conditional routing, cycles, and the complex control flows that real-world consulting agents require.